In [ ]:
import pandas as pd

# Path to data frame with WSIs
# df_path = r"D:\DATA\with_snomed_category.csv"
# df_path = r"D:\DATA\abmil_exp2.csv"
# df_path = r"D:\DATA\abmil_exp3.csv"
df_path = r"D:\DATA\abmil_inference_exp3.csv"

df_all = pd.read_csv(df_path)
print(df_all.columns)

# Paths for ABMIL inference output
zarr_dir = r"D:\NOTEBOOKS\Christine\all_slides\zarr"
checkpoint_path = r"D:\NOTEBOOKS\Christine\checkpoints\exp3_h-optimus-0\fold_1_auc_0.8883.pt"
cache_path = r"D:\NOTEBOOKS\Christine\checkpoints\exp3_h-optimus-0\fold_1_inference_cache.pkl"
xml_dir = r"D:\NOTEBOOKS\Christine\exp3\xml"

In [ ]:
from scripts.abmil_pipeline import DiseaseClassification

all_filenames = df_all['filename'].tolist()

classifier = DiseaseClassification(checkpoint_path=checkpoint_path, zarr_dir=zarr_dir, slides=all_filenames, cache_path = cache_path)
print('Classifier initialized for', len(all_filenames), 'slides')

In [ ]:
# Prepare true labels if available
true_labels = df_all.get('M_idx', None)
if true_labels is not None:
    true_labels = true_labels.tolist()

classifier.process_slides(true_labels=true_labels, top_k=100)

slide_cache = classifier._slide_cache

In [ ]:
# Get assessment report + confusion matrix  
report = classifier.assessment_report(true_labels=true_labels)

In [ ]:
from scripts.roi_selection import ROISelector

slide = all_filenames[0]  # Change index to select a different slide

# Select top_k ROIs 
roi_selector = ROISelector(cache_path=cache_path, slide_path=slide, top_k=100)

# Preserve the notebook variables used by the later ROI/Napari cells.
top_tiles_gdf = roi_selector.top_tiles_from_slide_data()
wsi = roi_selector.get_wsi()
sdata = roi_selector.get_sdata()
roi_polygons = roi_selector.napari_polygons()


In [ ]:
sdata

In [ ]:
from spatialdata.models import ShapesModel
sdata.shapes["ROIs"] = ShapesModel.parse(top_tiles_gdf)

In [ ]:
import napari 

# Run napari
viewer = napari.Viewer()

viewer.add_image(
    wsi, 
    name="slide", 
    multiscale=True
)

viewer.add_shapes(
    napari_polygons(),
    shape_type="polygon",
    edge_color="red",
    face_color="red",
    name="top_tiles",
)

napari.run()

In [ ]:
import numpy as np
# Add calibration points
points_layer = viewer.layers["calibration_points"]
calibration_points = points_layer.data
sdata.points["calibration_points"] = PointsModel.parse(
    np.array(calibration_points)
)

In [ ]:
import os
from dvpio.write.shapes import write_lmd

# CHECK IF FORMAT IS STILL COMPATIBLE

slide_base = os.path.basename(slide)
xml_path = os.path.join(xml_dir, slide_base)

H = sdata.images["image"].data.shape[1]
print(H)

affine_transformation = np.array([
    [1,  0, 0],
    [0, -1, H],
    [0,  0, 1]
])

write_lmd(
    xml_path,
    sdata.shapes["tiles"],
    calibration_points=sdata.points["calibration_points"],
    affine_transformation=affine_transformation
)
print(f"Saved xml file to {path_lmd}")